<div style="background-color:#F3F2EE">
    <br /><br />
        <p style="text-align: center;">
            <font size="6" color='#0A1781'>
                <strong>
                    Séries Temporais para Projetos de Data Science
                </strong>
              </font>
        </p>
        <p style="text-align: center;">
            <font size="6" color='#C58A1E'>
                <strong>
                    Projeto 02 - Previsão de vendas
                </strong>
            </font>
        </p>
        <p style="text-align: center;">
            <font size="5" color='#C58A1E'>
                <strong>
                    Data Understanding Estatístico (PySpark em Cluster Docker)
                </strong>
            </font>
        </p>
    <br />
</div>

<div style="background-color:#F3F2EE">
    <p style="text-align: right;">
      <font size="4" color='#444444'>
            Roberto SSoares - LfLngLrnng
      </font>
    </p>
    <p style="text-align: right;"><font size="2" color='#444444'>
        <a href="https://www.linkedin.com/in/roberto-dos-santos-soares/">in/roberto-dos-santos-soares</a><br /><a href="https://roberto-ssoares.github.io/meu-portfolio/">Portifólio: roberto-ssoares</a>
    </p>
    <p style="text-align: right;">
        <font size="4" color='#444444'>
            " [+] Faturamento,<br \> [-] Custo,<br \> [+] Qualidade de vida "
        </font>
        <br />
        <font size="2" color='#918e8e'>"Bruno Jardim"
        </font>
    </p>        
    <p style="text-align: right;">        
        <font size="2" color='#918e8e'>           
        </font>
    </p>
</div>

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>📌 Objetivo</strong></font>

<font size="2" color='#66666'>

Construir e avaliar um **baseline replicável** para previsão de vendas ao longo do tempo usando PySpark MLlib, com:

- Split **temporal** (sem leakage)
- Pipeline (VectorAssembler + LinearRegression)
- Avaliação (RMSE e R²)
- Persistência do modelo e previsões em **artifacts/**
- Evidências de execução distribuída via Spark UI / History Server

> Nota: baseline simples serve como diagnóstico para orientar evolução (lags, médias móveis, modelos não lineares).

---

</font></div>

<div style="background-color:#f3f2ee">
    
# <font size="5" color='#CC403E'><strong>✔️ SETUP (Instalação e Importação dos pacotes)</strong></font>

<font size="3" color='#66666'></font></div>

<div style="background-color:#f3f2ee">
    
## <font size="4" color='#FF5733'><strong>📚 Instalando e Carregando os Pacotes</strong></font>

<font size="3" color='#66666'></font></div>

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
#!pip install -q -U watermark

<div style="background-color:#f3f2ee">
    
## <font size="4" color='#FF5733'><strong>📚 Importando Bibliotecas</strong></font>

<font size="3" color='#66666'></font></div>

In [1]:
# Imports para manipulação de dados
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, isnan, lit, when, year, month, to_date, min as fmin, max as fmax
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Summarizer
from pyspark.ml.stat import Correlation

# Filtrando warnings
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "RobertoSSoares-LfLngLrnng"

Author: RobertoSSoares-LfLngLrnng



<div style="background-color:#f3f2ee">
    
# <font size="5" color='#CC403E'><strong>✔️ Business Understanding — Previsão de vendas</strong></font>

<font size="3" color='#66666'></font></div>

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>Observe:</strong></font>

<font size="2" color='#66666'>

- Pergunta de negócio implícita
    - **“Com base no histórico, conseguimos antecipar o comportamento das vendas ao longo do tempo para apoiar planejamento e decisão?”**

📌 Importante:
- Não se promete alta acurácia, promete previsibilidade inicial.
- Isso justifica um baseline simples.

---

</font></div>

<div style="background-color:#f3f2ee">
    
# <font size="5" color='#CC403E'><strong>✔️ Data Understanding - Estatística</strong></font>

<font size="3" color='#66666'></font></div>

<div style="background-color:#f3f2ee">
    
<font size="3" color='#CC403E'><strong>📌 PySpark em Cluster Docker</strong></font>

<font size="2" color='#66666'>

> **Objetivo desta etapa**
>    - Entender o comportamento das vendas ao longo do tempo, verificando:
>        - qualidade e consistência do dataset (schema, nulos, datas)
>        - tendência e sazonalidade (agregações por ano/mês)
>        - estatísticas descritivas (Summarizer)
>        - correlação entre variáveis temporais e vendas
>        - evidências para validar a estratégia de modelagem baseline (Year/Month → Sales)

> **Ambiente**
>    - Driver: Jupyter (container **jupyter-rss**)
>    - Cluster: Spark Master + 2 Workers (Docker)
>    - Event logs: habilitados (History Server)

---

</font></div>

<div style="background-color:#f3f2ee">
    
## <font size="3" color='#CC403E'><strong>📌 Conexão com o Spark (cluster)</strong></font>

<font size="2" color='#66666'>

- **Objetivo:** garantir que o notebook está operando como driver conectado ao cluster Spark (master + workers).  
- **Ação:** criar/obter **SparkSession** apontando para **spark://spark-master-rss:7077**.  
- **Justificativa técnica:** a validação de execução distribuída é parte do requisito de **ambiente realista** e permite inspecionar Spark UI/History.  
- **Resultado esperado:** SparkSession ativa e job simples executando (sanity check).

---

</font></div>

<div style="background-color:#f3f2ee">
    
### <font size="3" color='#CC403E'><strong>⚙️ SparkSession + sanity check (Code)</strong></font>

<font size="3" color='#66666'></font></div>

In [4]:
spark = (
    SparkSession.builder
    .appName("P02_Data_Understanding")
    .master("spark://spark-master-rss:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.sparkContext._jsc.sc().setLogLevel("ERROR")

In [5]:
# Sanity check (rápido)
_ = spark.range(1, 1_000_000).count()
print(_)
print("Spark conectado ao cluster com sucesso.")

[Stage 0:>                                                          (0 + 4) / 4]

999999
Spark conectado ao cluster com sucesso.


<div style="background-color:#f3f2ee">
    
## <font size="3" color='#CC403E'><strong>📌 Parâmetros do dataset</strong></font>

<font size="2" color='#66666'>

- **Objetivo:** padronizar caminhos e nomes de colunas.  
- **Ação:** definir caminho do dataset e colunas principais.  
- **Justificativa técnica:** reprodutibilidade e menor risco de hardcode no notebook.  
- **Resultado esperado:** variáveis definidas para leitura e validações.
---

</font></div>

<div style="background-color:#f3f2ee">
    
### <font size="3" color='#CC403E'><strong>⚙️ Parâmetros de entrada/saída</strong></font>

<font size="3" color='#66666'></font></div>

In [6]:
DATA_PATH = os.getenv("DATA_PATH", "/opt/spark/dados/dataset.csv")
DATE_COL = "Date"
TARGET_COL = "Sales"

print("DATA_PATH =", DATA_PATH)

DATA_PATH = /opt/spark/dados/dataset.csv


<div style="background-color:#f3f2ee">
    
## <font size="3" color='#CC403E'><strong>📌 Leitura do dataset</strong></font>

<font size="2" color='#66666'>

- **Objetivo:** carregar dados e validar schema.  
- **Ação:** leitura CSV com inferência de schema e header.  
- **Justificativa técnica:** schema correto evita erros silenciosos em cast e features temporais.  
- **Resultado esperado:** DataFrame carregado, schema consistente e amostra de linhas.
    
---

</font></div>

<div style="background-color:#f3f2ee">
    
### <font size="3" color='#CC403E'><strong>⚙️ Leitura + schema + sample (Code)</strong></font>

<font size="3" color='#66666'></font></div>

In [7]:
df_raw = spark.read.csv(DATA_PATH, header=True, inferSchema=True)

print(df_raw.printSchema())
print(df_raw.show(5, truncate=False))
print("Rows:", df_raw.count())

root
 |-- Date: timestamp (nullable = true)
 |-- Sales: integer (nullable = true)

None
+-------------------+-----+
|Date               |Sales|
+-------------------+-----+
|2013-01-01 00:00:00|113  |
|2013-02-01 00:00:00|119  |
|2013-03-01 00:00:00|134  |
|2013-04-01 00:00:00|129  |
|2013-05-01 00:00:00|121  |
+-------------------+-----+
only showing top 5 rows

None
Rows: 144


<div style="background-color:#f3f2ee">
    
## <font size="3" color='#CC403E'><strong>📌 Sanity checks (qualidade do dado)</strong></font>

<font size="2" color='#66666'>

- **Objetivo:** entender se existem nulos, tipos inconsistentes e datas inválidas.  
- **Ações:**
    - contagem de nulos por coluna
    - validação de datas (cast para date)
- **Justificativa técnica:** problemas em **Date** afetam diretamente agregações e split temporal; problemas em **Sales** impactam métricas e regressão.  
- **Resultado esperado:** visão clara de nulos/invalidos e plano de correção.
    
---

</font></div>

<div style="background-color:#f3f2ee">
    
### <font size="3" color='#CC403E'><strong>⚙️ Nulos por coluna (Code)</strong></font>

<font size="3" color='#66666'></font></div>

In [8]:
# Nulos (inclui NaN para colunas numéricas)
#from pyspark.sql.functions import col, count, when, isnan, lit
#from functools import reduce

dtypes = dict(df_raw.dtypes)

exprs = []
for c in df_raw.columns:
    is_null = col(c).isNull()

    # se for numérica, inclui isnan(col(c)); senão, usa lit(False) (ainda é Column)
    is_nan = isnan(col(c)) if dtypes.get(c) in ("double", "float") else lit(False)

    exprs.append(count(when(is_null | is_nan, 1)).alias(c))

nulls_df = df_raw.select(exprs)
nulls_df.show(truncate=False)

+----+-----+
|Date|Sales|
+----+-----+
|0   |0    |
+----+-----+



In [9]:
df_raw.printSchema()

root
 |-- Date: timestamp (nullable = true)
 |-- Sales: integer (nullable = true)



<div style="background-color:#f3f2ee">
    
## <font size="3" color='#CC403E'><strong>📌 Padronização de data e criação de features temporais (Year, Month)</strong></font>

<font size="2" color='#66666'>

- **Objetivo:** garantir que **Date** é uma data válida e criar variáveis temporais.  
- **Ação:** cast para **date**, derivar **Year** e **Month**.  
- **Justificativa técnica:** Year/Month são a base do baseline e suportam análises de tendência/sazonalidade.  
- **Resultado esperado:** DataFrame com colunas **Date**, **Year**, **Month** prontas para exploração.
    
---

</font></div>

<div style="background-color:#f3f2ee">
    
### <font size="3" color='#CC403E'><strong>⚙️ Transformação temporal (Code)</strong></font>

<font size="3" color='#66666'></font></div>

In [10]:
df = df_raw.withColumn(DATE_COL, to_date(col(DATE_COL))) \
           .withColumn("Year", year(col(DATE_COL))) \
           .withColumn("Month", month(col(DATE_COL)))

df.select(DATE_COL, "Year", "Month", TARGET_COL).show(5, truncate=False)

df.select(
    fmin(col(DATE_COL)).alias("min_date"),
    fmax(col(DATE_COL)).alias("max_date")
).show()

+----------+----+-----+-----+
|Date      |Year|Month|Sales|
+----------+----+-----+-----+
|2013-01-01|2013|1    |113  |
|2013-02-01|2013|2    |119  |
|2013-03-01|2013|3    |134  |
|2013-04-01|2013|4    |129  |
|2013-05-01|2013|5    |121  |
+----------+----+-----+-----+
only showing top 5 rows

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2013-01-01|2024-12-01|
+----------+----------+



<div style="background-color:#f3f2ee">
    
## <font size="3" color='#CC403E'><strong>📌 Perfil básico de vendas</strong></font>

<font size="2" color='#66666'>

- **Objetivo:** entender escala e distribuição inicial do alvo (**Sales**).  
- **Ação:** descrever estatísticas e checar outliers potenciais.  
- **Justificativa técnica:** ajuda a interpretar RMSE depois e identificar necessidade de transformação/robustez.  
- **Resultado esperado:** estatísticas gerais e sinais de variabilidade.
    
---

</font></div>

<div style="background-color:#f3f2ee">
    
### <font size="3" color='#CC403E'><strong>⚙️ Estatísticas simples (Code)</strong></font>

<font size="3" color='#66666'></font></div>

In [11]:
df.select(TARGET_COL).describe().show()

+-------+------------------+
|summary|             Sales|
+-------+------------------+
|  count|               144|
|   mean|280.34027777777777|
| stddev|119.91728869764205|
|    min|               104|
|    max|               622|
+-------+------------------+



<div style="background-color:#f3f2ee">
    
## <font size="3" color='#CC403E'><strong>📌 Estatísticas avançadas (Summarizer)</strong></font>

<font size="2" color='#66666'>

- **Objetivo:** obter métricas como média, variância, min e max do alvo.  
- **Ação:** usar **pyspark.ml.stat.Summarizer**.  
- **Justificativa técnica:** sumarização formal para report e consistência com MLlib.  
- **Resultado esperado:** métricas consolidadas para **Sales**.    

---

</font></div>

<div style="background-color:#f3f2ee">
    
### <font size="3" color='#CC403E'><strong>⚙️ Summarizer (Code)</strong></font>

<font size="3" color='#66666'></font></div>

In [12]:
#from pyspark.ml.feature import VectorAssembler
#from pyspark.ml.stat import Summarizer
#from pyspark.sql.functions import col

assembler = VectorAssembler(inputCols=[TARGET_COL], outputCol="sales_vec")
df_vec = assembler.transform(df.select(TARGET_COL).dropna())

summary = df_vec.select(
    Summarizer.metrics("mean", "variance", "min", "max").summary(col("sales_vec")).alias("summary")
)

summary.show(truncate=False)

+-------------------------------------------------------------+
|summary                                                      |
+-------------------------------------------------------------+
|{[280.34027777777777], [14380.15612859363], [104.0], [622.0]}|
+-------------------------------------------------------------+



<div style="background-color:#f3f2ee">
    
### <font size="3" color='#CC403E'><strong>⚙️ Estatísticas avançadas "SQL Aggregations" (Code)</strong></font>

<font size="3" color='#66666'></font></div>

In [13]:
from pyspark.sql.functions import avg, variance, min as fmin, max as fmax

df.select(
    avg(col(TARGET_COL)).alias("mean"),
    variance(col(TARGET_COL)).alias("variance"),
    fmin(col(TARGET_COL)).alias("min"),
    fmax(col(TARGET_COL)).alias("max"),
).show()


+------------------+-----------------+---+---+
|              mean|         variance|min|max|
+------------------+-----------------+---+---+
|280.34027777777777|14380.15612859363|104|622|
+------------------+-----------------+---+---+



<div style="background-color:#f3f2ee">
    
## <font size="3" color='#CC403E'><strong>📌 Agregações por tempo (sazonalidade e tendência)</strong></font>

<font size="2" color='#66666'>

- **Objetivo:** avaliar evidências de sazonalidade (mês) e tendência (ano).  
- **Ações:**
    - média e soma de vendas por mês
    - média e soma de vendas por ano
- **Justificativa técnica:** se houver diferenças sistemáticas por mês, **Month** tende a carregar sinal; se houver mudança por ano, **Year** pode capturar tendência.  
- **Resultado esperado:** tabelas ordenadas com comportamento temporal.

---

</font></div>

<div style="background-color:#f3f2ee">
    
### <font size="3" color='#CC403E'><strong>⚙️ Por mês (Code)</strong></font>

<font size="3" color='#66666'></font></div>

In [14]:
from pyspark.sql.functions import avg, sum as fsum, count

by_month = (
    df.groupBy("Month")
      .agg(
          count("*").alias("n"),
          avg(col("Sales")).alias("avg_sales"),
          fsum(col("Sales")).alias("sum_sales"),
      )
      .orderBy("Month")
)

by_month.show(12, truncate=False)

+-----+---+------------------+---------+
|Month|n  |avg_sales         |sum_sales|
+-----+---+------------------+---------+
|1    |12 |241.91666666666666|2903     |
|2    |12 |235.08333333333334|2821     |
|3    |12 |270.3333333333333 |3244     |
|4    |12 |267.0833333333333 |3205     |
|5    |12 |271.8333333333333 |3262     |
|6    |12 |311.6666666666667 |3740     |
|7    |12 |351.3333333333333 |4216     |
|8    |12 |351.0833333333333 |4213     |
|9    |12 |302.4166666666667 |3629     |
|10   |12 |266.5833333333333 |3199     |
|11   |12 |232.91666666666666|2795     |
|12   |12 |261.8333333333333 |3142     |
+-----+---+------------------+---------+



<div style="background-color:#f3f2ee">
    
### <font size="3" color='#CC403E'><strong>⚙️ Por ano (Code)</strong></font>

<font size="3" color='#66666'></font></div>

In [15]:
from pyspark.sql.functions import avg, sum as fsum, count

by_year = (
    df.groupBy("Year")
      .agg(
          count("*").alias("n"),
          avg(col("Sales")).alias("avg_sales"),
          fsum(col("Sales")).alias("sum_sales"),
      )
      .orderBy("Year")
)

by_year.show(100, truncate=False)

+----+---+------------------+---------+
|Year|n  |avg_sales         |sum_sales|
+----+---+------------------+---------+
|2013|12 |127.0             |1524     |
|2014|12 |139.66666666666666|1676     |
|2015|12 |170.16666666666666|2042     |
|2016|12 |197.16666666666666|2366     |
|2017|12 |225.0             |2700     |
|2018|12 |238.91666666666666|2867     |
|2019|12 |284.0             |3408     |
|2020|12 |328.25            |3939     |
|2021|12 |368.4166666666667 |4421     |
|2022|12 |381.0             |4572     |
|2023|12 |428.3333333333333 |5140     |
|2024|12 |476.1666666666667 |5714     |
+----+---+------------------+---------+



<div style="background-color:#f3f2ee">
    
## <font size="3" color='#CC403E'><strong>📌 Hipóteses (evidência por agrupamento)</strong></font>

<font size="2" color='#66666'>

**Hipótese de sazonalidade:**
- **H0**: não há diferença relevante no nível médio de vendas entre meses.
- **Evidência empírica**: variações consistentes em **avg_sales** por mês sugerem sazonalidade.

**Hipótese de tendência:**
- **H0**: não há mudança relevante de vendas entre anos.
- **Evidência empírica**: variações em **avg_sales** e/ou **sum_sales** ao longo dos anos sugerem tendência.

> Nota: aqui usamos evidência empírica (EDA) como base para justificar features. Testes formais podem ser adicionados em uma evolução do projeto.

---

</font></div>

<div style="background-color:#f3f2ee">
    
## <font size="3" color='#CC403E'><strong>📌 Correlação (Year, Month, Sales)</strong></font>

<font size="2" color='#66666'>

- **Objetivo:** medir se existe relação linear entre variáveis temporais e vendas.  
- **Ação:** gerar matriz de correlação para **[Year, Month, Sales]**.  
- **Justificativa técnica:** se a correlação linear for baixa, um baseline com regressão linear tende a ter R² baixo, reforçando a necessidade de features com memória temporal (lags / médias móveis).  
- **Resultado esperado:** matriz de correlação interpretável.

---

</font></div>

<div style="background-color:#f3f2ee">
    
### <font size="3" color='#CC403E'><strong>⚙️ Correlação (Code)</strong></font>

<font size="3" color='#66666'></font></div>

In [16]:
#from pyspark.ml.stat import Correlation
#from pyspark.ml.feature import VectorAssembler

vec = VectorAssembler(inputCols=["Year", "Month", TARGET_COL], outputCol="features_corr")
df_corr = vec.transform(df.select("Year", "Month", TARGET_COL).dropna()).select("features_corr")

corr = Correlation.corr(df_corr, "features_corr").head()[0]  # DenseMatrix
corr

DenseMatrix(3, 3, [1.0, 0.0, 0.9217, 0.0, 1.0, 0.0634, 0.9217, 0.0634, 1.0], False)

<div style="background-color:#f3f2ee">
    
<font size="3" color='#CC403E'><strong>📌 Interpretação esperada</strong></font>

<font size="2" color='#66666'>

- **Correlação Month ↔ Sales**: sinal de sazonalidade (se diferente de 0)
- **Correlação Year ↔ Sales**: sinal de tendência (se diferente de 0)
- Se ambas forem fracas, é esperado que um baseline linear com Year/Month tenha R² baixo.

> Resultado deste notebook deve ser coerente com o Notebook 2: o baseline funciona como diagnóstico.


---

</font></div>

<div style="background-color:#f3f2ee">
    
## <font size="3" color='#CC403E'><strong>📌 Checklist de consistência para o baseline (Year/Month → Sales)</strong></font>

<font size="2" color='#66666'>

Antes de modelar, confirme:
- **Date** está em formato date (sem nulos relevantes)
- **Sales** é numérico e sem nulos relevantes
- **Year** e **Month** foram derivados corretamente
- há evidência de padrão por tempo (mesmo que fraca)

Se o sinal temporal for fraco, a próxima evolução é incluir:
- **lag_1**, **lag_2**, **lag_3**… (memória)
- médias móveis (janela 3/6/12)
- variáveis exógenas (se existirem)

---

</font></div>

<div style="background-color:#f3f2ee">
    
## <font size="3" color='#CC403E'><strong>📌 Conclusões e próximos passos</strong></font>

<font size="2" color='#66666'>

**Conclusão desta etapa (Data Understanding):**
- Identificamos a qualidade do dataset e a faixa temporal disponível.
- Quantificamos a variabilidade de **Sales**.
- Analisamos evidências de sazonalidade/tendência por agregações temporais.
- Avaliamos correlação linear entre tempo e vendas, justificando o baseline e orientando melhorias.

**Próximos passos (Data Preparation / Modeling):**
- Criar features com memória temporal (lags).
- Incluir médias móveis com window functions.
- Reexecutar baseline e comparar ganhos.
- Comparar modelos (Linear vs GBT/RF).

---

</font></div>

<div style="background-color:#f3f2ee">
    
### <font size="3" color='#CC403E'><strong>⚙️ Fechar Spark (Code)</strong></font>

<font size="3" color='#66666'></font></div>

In [17]:
spark.stop()

<div style="background-color:#f3f2ee">
    
<font size="6" color='#CC403E'><strong>Fim</strong></font>

<font size="3" color='#66666'></font></div>

In [18]:
#!pip install nbconvert -U -q
!jupyter nbconvert P02_01_Data_Understanding_Estatistico_PySpark.ipynb --to html --template my-template-html-v07.tpl

[NbConvertApp] Converting notebook P02_01_Data_Understanding_Estatistico_PySpark.ipynb to html
[NbConvertApp] Writing 42599 bytes to P02_01_Data_Understanding_Estatistico_PySpark.html
